In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Heterogeneous Network for Drug Repurposing
==========================================
This script builds a heterogeneous network from:
    - Protein‑protein interactions (from oncogene_pathways.xml)
    - Drug‑target associations (from drug mechanism of action data)
    - Disease‑gene associations (from Open Targets data)

It then applies supervised (Node2Vec+LR/RF, RGCN) and unsupervised
(path counting, weighted PageRank) methods to predict drug‑disease
associations, specifically for breast cancer, and validates predictions
with PubMed.

Usage:
    python 08_drug_disease_heterogeneous_network.py
"""

import os
import random
import time
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
import xml.etree.ElementTree as ET
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
import dgl
from dgl.nn import HeteroGraphConv, GraphConv
from node2vec import Node2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from Bio import Entrez

# Set random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# ----------------------------------------------------------------------
# 1. Build protein graph from XML
# ----------------------------------------------------------------------
def build_protein_graph(xml_file):
    """Parse XML and return a directed graph."""
    G = nx.DiGraph()
    tree = ET.parse(xml_file)
    root = tree.getroot()
    for pathway in root.findall('pathway'):
        node_list = pathway.find('nodeList')
        if node_list is not None:
            for node in node_list.findall('node'):
                node_name = node.get('name')
                if node_name:
                    G.add_node(node_name)
        edge_list = pathway.find('edgeList')
        if edge_list is not None:
            for edge in edge_list.findall('edge'):
                start = edge.find('startNode').get('node')
                end = edge.find('endNode').get('node')
                if start and end:
                    G.add_edge(start, end)
    return G

# ----------------------------------------------------------------------
# 2. Load external data (Open Targets, ChEMBL)
# ----------------------------------------------------------------------
def load_target_data(target_dir):
    """Load target (gene) data from parquet files."""
    target_dir = Path(target_dir)
    target_df = pd.concat([pd.read_parquet(f) for f in target_dir.glob("*.parquet")], ignore_index=True)
    target_df = target_df[['id', 'approvedSymbol']].drop_duplicates()
    target_df.rename(columns={'approvedSymbol': 'symbol'}, inplace=True)
    return target_df

def load_disease_data(disease_dir):
    """Load disease data from parquet files."""
    disease_dir = Path(disease_dir)
    disease_df = pd.concat([pd.read_parquet(f) for f in disease_dir.glob("*.parquet")], ignore_index=True)
    disease_df = disease_df[['id', 'name']].drop_duplicates()
    return disease_df

def load_drug_data(drug_mol_dir):
    """Load drug molecule data."""
    drug_mol_dir = Path(drug_mol_dir)
    drug_mol_df = pd.concat([pd.read_parquet(f) for f in drug_mol_dir.glob("*.parquet")], ignore_index=True)
    drug_mol_df = drug_mol_df[['id', 'name']].drop_duplicates()
    return drug_mol_df

def load_drug_target_associations(moa_dir, target_df, drug_mol_df):
    """Load drug‑target associations from mechanism of action files."""
    moa_dir = Path(moa_dir)
    moa_df = pd.concat([pd.read_parquet(f) for f in moa_dir.glob("*.parquet")], ignore_index=True)
    moa_df = moa_df.dropna(subset=['chemblIds', 'targets'])

    records = []
    for _, row in moa_df.iterrows():
        chembl_ids = row['chemblIds']
        targets = row['targets']
        if not isinstance(chembl_ids, (list, np.ndarray)):
            chembl_ids = [chembl_ids]
        if not isinstance(targets, (list, np.ndarray)):
            targets = [targets]
        if len(chembl_ids) == len(targets):
            for cid, tid in zip(chembl_ids, targets):
                records.append({'chembl_id': cid, 'target_id': tid})
        else:
            for cid in chembl_ids:
                for tid in targets:
                    records.append({'chembl_id': cid, 'target_id': tid})

    moa_expanded = pd.DataFrame(records)
    moa_expanded = moa_expanded.merge(drug_mol_df, left_on='chembl_id', right_on='id', how='inner')
    moa_expanded = moa_expanded.merge(target_df, left_on='target_id', right_on='id', how='inner')
    return moa_expanded[['symbol', 'name']].drop_duplicates()

def load_disease_gene_associations(assoc_dir, target_ids, target_df, disease_df, score_threshold=0.1):
    """Load disease‑gene associations and filter by target IDs and score."""
    assoc_dir = Path(assoc_dir)
    assoc_list = []
    for f in assoc_dir.glob("*.parquet"):
        df_chunk = pd.read_parquet(f, columns=['targetId', 'diseaseId', 'associationScore'])
        df_filtered = df_chunk[df_chunk['targetId'].isin(target_ids)]
        if not df_filtered.empty:
            assoc_list.append(df_filtered)
    if not assoc_list:
        return pd.DataFrame(columns=['symbol', 'name', 'associationScore'])
    assoc_df = pd.concat(assoc_list, ignore_index=True)
    assoc_df = assoc_df.merge(target_df[['id', 'symbol']], left_on='targetId', right_on='id', how='inner')
    assoc_df = assoc_df.merge(disease_df[['id', 'name']], left_on='diseaseId', right_on='id', how='inner')
    assoc_df = assoc_df[assoc_df['associationScore'] >= score_threshold]
    return assoc_df

# ----------------------------------------------------------------------
# 3. Build heterogeneous network
# ----------------------------------------------------------------------
def build_heterogeneous_network(protein_graph, gene_to_drugs, gene_to_diseases):
    H = nx.DiGraph()
    H.add_nodes_from(protein_graph.nodes(data=True))
    H.add_edges_from(protein_graph.edges(data=True))

    # Add drug nodes and edges (protein -> drug)
    for protein, drugs in gene_to_drugs.items():
        if protein in H:
            for drug in drugs:
                H.add_node(drug, type='drug')
                H.add_edge(protein, drug, relation='targeted_by')

    # Add disease nodes and edges (protein -> disease)
    for protein, diseases in gene_to_diseases.items():
        if protein in H:
            for disease in diseases:
                H.add_node(disease, type='disease')
                H.add_edge(protein, disease, relation='associated_with')
    return H

# ----------------------------------------------------------------------
# 4. Supervised methods (Node2Vec + LR/RF, RGCN)
# ----------------------------------------------------------------------
def train_node2vec(H, dimensions=128, walk_length=30, num_walks=200, workers=4):
    H_und = H.to_undirected()
    node2vec = Node2Vec(H_und, dimensions=dimensions, walk_length=walk_length,
                        num_walks=num_walks, workers=workers)
    model = node2vec.fit(window=10, min_count=1, batch_words=4)
    return model

def prepare_supervised_data(H, gene_to_drugs, gene_to_diseases, test_size=0.2):
    """Build positive and negative drug‑disease pairs with degree‑matched negative sampling."""
    # Build drug->genes and disease->genes maps
    drug_to_genes = defaultdict(set)
    for gene, drugs in gene_to_drugs.items():
        for drug in drugs:
            drug_to_genes[drug].add(gene)

    disease_to_genes = defaultdict(set)
    for gene, diseases in gene_to_diseases.items():
        for disease in diseases:
            disease_to_genes[disease].add(gene)

    positive_pairs = []
    for drug, genes_drug in drug_to_genes.items():
        for disease, genes_dis in disease_to_genes.items():
            if genes_drug & genes_dis:
                positive_pairs.append((drug, disease))
    positive_pairs = list(set(positive_pairs))
    print(f"Positive pairs: {len(positive_pairs)}")

    # Degree buckets for negative sampling
    H_und = H.to_undirected()
    degree = dict(H_und.degree())
    degree_vals = np.array(list(degree.values()))
    bins = np.percentile(degree_vals, [25, 50, 75])
    def bucket(d):
        if d <= bins[0]:
            return 0
        elif d <= bins[1]:
            return 1
        elif d <= bins[2]:
            return 2
        else:
            return 3

    pos_bucket_dist = defaultdict(int)
    for drug, disease in positive_pairs:
        pos_bucket_dist[(bucket(degree.get(drug, 0)), bucket(degree.get(disease, 0)))] += 1

    bucket_nodes = defaultdict(list)
    for node, d in degree.items():
        bucket_nodes[bucket(d)].append(node)

    existing_pairs_set = set(positive_pairs)
    negative_pairs = []
    for (b_drug, b_dis), count in pos_bucket_dist.items():
        candidates_drug = bucket_nodes[b_drug]
        candidates_dis = bucket_nodes[b_dis]
        sampled = 0
        attempts = 0
        while sampled < count and attempts < count * 20:
            drug = random.choice(candidates_drug)
            dis = random.choice(candidates_dis)
            if drug != dis and (drug, dis) not in existing_pairs_set:
                negative_pairs.append((drug, dis))
                sampled += 1
            attempts += 1
    print(f"Negative pairs: {len(negative_pairs)}")

    all_pairs = positive_pairs + negative_pairs
    labels = [1] * len(positive_pairs) + [0] * len(negative_pairs)
    return train_test_split(all_pairs, labels, test_size=test_size, stratify=labels, random_state=RANDOM_STATE)

def build_rgcn_graph(H, drug_emb, dis_emb):
    """Convert heterogeneous network to DGL graph with initial features."""
    node_types = {}
    for node, attr in H.nodes(data=True):
        ntype = attr.get('type', 'protein')
        node_types.setdefault(ntype, []).append(node)

    node_id = {}
    for ntype, nodes in node_types.items():
        for i, n in enumerate(nodes):
            node_id[n] = (ntype, i)

    edge_dict = {}
    for u, v in H.edges():
        utype = H.nodes[u].get('type', 'protein')
        vtype = H.nodes[v].get('type', 'protein')
        rel = 'interact'
        edge_dict.setdefault((utype, rel, vtype), []).append((node_id[u][1], node_id[v][1]))

    graph_data = {}
    for (stype, rel, dtype), edges in edge_dict.items():
        graph_data[(stype, rel, dtype)] = (torch.tensor([e[0] for e in edges]), torch.tensor([e[1] for e in edges]))

    g = dgl.heterograph(graph_data)

    # Initialize node features (Node2Vec embeddings if available, else random)
    for ntype in g.ntypes:
        g.nodes[ntype].data['h'] = torch.randn(g.num_nodes(ntype), 128)
    return g, node_id

class RGCN(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim, rel_names):
        super().__init__()
        self.conv1 = HeteroGraphConv({rel: GraphConv(in_dim, hid_dim) for rel in rel_names})
        self.conv2 = HeteroGraphConv({rel: GraphConv(hid_dim, out_dim) for rel in rel_names})

    def forward(self, g, h_dict):
        h = self.conv1(g, h_dict)
        h = {k: torch.relu(v) for k, v in h.items()}
        h = self.conv2(g, h)
        return h

# ----------------------------------------------------------------------
# 5. Unsupervised methods
# ----------------------------------------------------------------------
def path_counting_recommendation(H, disease):
    """Recommend drugs by counting shared proteins."""
    if disease not in H:
        return []
    proteins = list(H.predecessors(disease))
    drugs = []
    for p in proteins:
        for nbr in H.successors(p):
            if H.nodes[nbr].get('type') == 'drug':
                drugs.append(nbr)
    return Counter(drugs).most_common()

def weighted_pagerank_recommendation(H, disease, gene_to_disease_scores, alpha=0.85):
    """Personalized PageRank with edge weights from disease‑gene scores."""
    H_weighted = nx.Graph()
    # PPI edges weight 1.0
    for u, v in H.edges():
        if H.nodes[u].get('type') == 'protein' and H.nodes[v].get('type') == 'protein':
            H_weighted.add_edge(u, v, weight=1.0)
    # Drug‑protein edges weight 1.0
    for u, v in H.edges():
        if H.nodes[u].get('type') == 'protein' and H.nodes[v].get('type') == 'drug':
            H_weighted.add_edge(u, v, weight=1.0)
    # Disease‑protein edges weight = association score
    for gene, disease_scores in gene_to_disease_scores.items():
        for dis, score in disease_scores:
            if dis == disease:
                H_weighted.add_edge(gene, dis, weight=score)

    # Personalization vector
    personalization = {node: 1 if node == disease else 0 for node in H_weighted.nodes()}
    ppr = nx.pagerank(H_weighted, alpha=alpha, personalization=personalization, weight='weight')
    drugs = [n for n, attr in H.nodes(data=True) if attr.get('type') == 'drug']
    return sorted([(d, ppr.get(d, 0)) for d in drugs], key=lambda x: x[1], reverse=True)

# ----------------------------------------------------------------------
# 6. Main pipeline
# ----------------------------------------------------------------------
def main():
    # ------------------------------------------------------------------
    # Data paths (adjust as needed)
    # ------------------------------------------------------------------
    xml_file = "oncogene_pathways.xml"
    target_dir = "target"
    disease_dir = "disease"
    drug_mol_dir = "drug_molecule"
    moa_dir = "drug_mechanism_of_action"
    assoc_dir = "association_overall_direct"

    # ------------------------------------------------------------------
    # Load data
    # ------------------------------------------------------------------
    print("Building protein graph...")
    G = build_protein_graph(xml_file)
    print(f"Protein graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    print("Loading target data...")
    target_df = load_target_data(target_dir)
    print(f"Target table size: {len(target_df)}")

    print("Loading disease data...")
    disease_df = load_disease_data(disease_dir)
    print(f"Disease table size: {len(disease_df)}")

    print("Loading drug data...")
    drug_mol_df = load_drug_data(drug_mol_dir)
    print(f"Drug molecule table size: {len(drug_mol_df)}")

    # Filter targets that appear in the protein graph
    protein_names = list(G.nodes())
    target_id_map = target_df[target_df['symbol'].isin(protein_names)]
    target_ids = set(target_id_map['id'])
    print(f"Proteins in graph mapped to {len(target_ids)} Ensembl IDs")

    # Drug‑target associations
    print("Loading drug‑target associations...")
    drug_target = load_drug_target_associations(moa_dir, target_df, drug_mol_df)
    gene_to_drugs = drug_target.groupby('symbol')['name'].apply(lambda x: list(set(x))).to_dict()
    print(f"Genes with drug associations: {len(gene_to_drugs)}")

    # Disease‑gene associations
    print("Loading disease‑gene associations...")
    assoc_filtered = load_disease_gene_associations(assoc_dir, target_ids, target_df, disease_df, score_threshold=0.1)
    gene_to_diseases = assoc_filtered.groupby('symbol')['name'].apply(lambda x: list(set(x))).to_dict()
    print(f"Genes with disease associations: {len(gene_to_diseases)}")

    # Build heterogeneous network
    print("Building heterogeneous network...")
    H = build_heterogeneous_network(G, gene_to_drugs, gene_to_diseases)
    print(f"Heterogeneous network: {H.number_of_nodes()} nodes, {H.number_of_edges()} edges")

    # ------------------------------------------------------------------
    # Supervised learning data
    # ------------------------------------------------------------------
    print("Preparing supervised data...")
    X_train, X_test, y_train, y_test = prepare_supervised_data(H, gene_to_drugs, gene_to_diseases)

    # Train Node2Vec
    print("Training Node2Vec...")
    node2vec_model = train_node2vec(H)
    drug_emb = {node: node2vec_model.wv[node] for node in H.nodes() if H.nodes[node].get('type') == 'drug'}
    dis_emb = {node: node2vec_model.wv[node] for node in H.nodes() if H.nodes[node].get('type') == 'disease'}

    # Build feature vectors (Hadamard product)
    def hadamard(drug, disease):
        return drug_emb[drug] * dis_emb[disease]

    X_train_vec = np.array([hadamard(d, dis) for d, dis in X_train if d in drug_emb and dis in dis_emb])
    y_train_vec = np.array([y_train[i] for i, (d, dis) in enumerate(X_train) if d in drug_emb and dis in dis_emb])
    X_test_vec = np.array([hadamard(d, dis) for d, dis in X_test if d in drug_emb and dis in dis_emb])
    y_test_vec = np.array([y_test[i] for i, (d, dis) in enumerate(X_test) if d in drug_emb and dis in dis_emb])

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    lr.fit(X_train_vec, y_train_vec)
    y_pred_lr = lr.predict_proba(X_test_vec)[:, 1]
    auc_lr = roc_auc_score(y_test_vec, y_pred_lr)
    aupr_lr = average_precision_score(y_test_vec, y_pred_lr)
    print(f"Logistic Regression: AUC={auc_lr:.4f}, AUPR={aupr_lr:.4f}")

    # Random Forest
    rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train_vec, y_train_vec)
    y_pred_rf = rf.predict_proba(X_test_vec)[:, 1]
    auc_rf = roc_auc_score(y_test_vec, y_pred_rf)
    aupr_rf = average_precision_score(y_test_vec, y_pred_rf)
    print(f"Random Forest: AUC={auc_rf:.4f}, AUPR={aupr_rf:.4f}")

    # ------------------------------------------------------------------
    # RGCN
    # ------------------------------------------------------------------
    print("Building RGCN graph...")
    g, node_id = build_rgcn_graph(H, drug_emb, dis_emb)
    # Prepare RGCN training data
    drug_ids = {d: node_id[d][1] for d in drug_emb if d in node_id}
    dis_ids = {d: node_id[d][1] for d in dis_emb if d in node_id}
    train_pairs = [(drug_ids[d], dis_ids[dis]) for d, dis in X_train if d in drug_ids and dis in dis_ids]
    train_labels = torch.tensor([y_train[i] for i, (d, dis) in enumerate(X_train) if d in drug_ids and dis in dis_ids], dtype=torch.float)
    test_pairs = [(drug_ids[d], dis_ids[dis]) for d, dis in X_test if d in drug_ids and dis in dis_ids]
    test_labels = torch.tensor([y_test[i] for i, (d, dis) in enumerate(X_test) if d in drug_ids and dis in dis_ids], dtype=torch.float)

    rgcn = RGCN(in_dim=128, hid_dim=64, out_dim=32, rel_names=g.etypes)
    optimizer = optim.Adam(rgcn.parameters(), lr=0.01)

    def rgcn_loss(model, g, pos_pairs):
        h = model(g, {nt: g.nodes[nt].data['h'] for nt in g.ntypes})
        drug_embeds = h['drug'][pos_pairs[:, 0]]
        dis_embeds = h['disease'][pos_pairs[:, 1]]
        pos_scores = (drug_embeds * dis_embeds).sum(1)
        neg_drugs = torch.randint(0, g.num_nodes('drug'), (len(pos_pairs),))
        neg_dis = torch.randint(0, g.num_nodes('disease'), (len(pos_pairs),))
        neg_scores = (h['drug'][neg_drugs] * h['disease'][neg_dis]).sum(1)
        return nn.BCEWithLogitsLoss()(torch.cat([pos_scores, neg_scores]),
                                      torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)]))

    print("Training RGCN...")
    for epoch in range(50):
        pos_indices = torch.tensor(train_pairs)
        loss = rgcn_loss(rgcn, g, pos_indices)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            print(f"Epoch {epoch}, loss={loss.item():.4f}")

    rgcn.eval()
    with torch.no_grad():
        h = rgcn(g, {nt: g.nodes[nt].data['h'] for nt in g.ntypes})
        test_scores = []
        for d_id, dis_id in test_pairs:
            score = (h['drug'][d_id] * h['disease'][dis_id]).sum().item()
            test_scores.append(score)
        auc_rgcn = roc_auc_score(test_labels.numpy(), test_scores)
        aupr_rgcn = average_precision_score(test_labels.numpy(), test_scores)
        print(f"RGCN: AUC={auc_rgcn:.4f}, AUPR={aupr_rgcn:.4f}")

    # ------------------------------------------------------------------
    # Unsupervised methods for breast cancer
    # ------------------------------------------------------------------
    target_disease = "breast cancer"
    disease_nodes = [n for n, attr in H.nodes(data=True) if attr.get('type') == 'disease']
    matched_disease = None
    for dn in disease_nodes:
        if dn.lower() == target_disease.lower():
            matched_disease = dn
            break
    if matched_disease is None:
        for dn in disease_nodes:
            if target_disease.lower() in dn.lower():
                matched_disease = dn
                break
    if matched_disease is None:
        raise ValueError(f"Disease '{target_disease}' not found in graph")
    print(f"Using disease node: {matched_disease}")

    # Path counting
    path_counts = dict(path_counting_recommendation(H, matched_disease))

    # Weighted PageRank
    # Build gene->disease score dictionary
    gene_to_disease_scores = defaultdict(list)
    for _, row in assoc_filtered.iterrows():
        gene_to_disease_scores[row['symbol']].append((row['name'], row['associationScore']))
    ppr_ranked = weighted_pagerank_recommendation(H, matched_disease, gene_to_disease_scores)
    ppr_scores = dict(ppr_ranked)

    # ------------------------------------------------------------------
    # Collect candidate drugs from supervised methods
    # ------------------------------------------------------------------
    all_drugs = [n for n, attr in H.nodes(data=True) if attr.get('type') == 'drug']
    # LR scores
    lr_scores = {}
    if matched_disease in dis_emb:
        dis_vec = dis_emb[matched_disease]
        for drug in all_drugs:
            if drug in drug_emb:
                feat = (drug_emb[drug] * dis_vec).reshape(1, -1)
                lr_scores[drug] = lr.predict_proba(feat)[0, 1]
            else:
                lr_scores[drug] = 0.0
    else:
        print("Disease not in dis_emb, skipping LR predictions")

    # RF scores
    rf_scores = {}
    if matched_disease in dis_emb:
        dis_vec = dis_emb[matched_disease]
        for drug in all_drugs:
            if drug in drug_emb:
                feat = (drug_emb[drug] * dis_vec).reshape(1, -1)
                rf_scores[drug] = rf.predict_proba(feat)[0, 1]
            else:
                rf_scores[drug] = 0.0

    # RGCN scores
    rgcn_scores = {}
    if 'rgcn' in dir():
        dis_idx = None
        for node, (ntype, idx) in node_id.items():
            if node == matched_disease and ntype == 'disease':
                dis_idx = idx
                break
        if dis_idx is not None:
            with torch.no_grad():
                h = rgcn(g, {nt: g.nodes[nt].data['h'] for nt in g.ntypes})
                dis_vec = h['disease'][dis_idx]
                drug_indices = []
                drug_names = []
                for node, (ntype, idx) in node_id.items():
                    if ntype == 'drug':
                        drug_names.append(node)
                        drug_indices.append(idx)
                scores = torch.mv(h['drug'][drug_indices], dis_vec).cpu().numpy()
                rgcn_scores = {drug: scores[i] for i, drug in enumerate(drug_names)}
        else:
            print("Disease index not found for RGCN")

    # ------------------------------------------------------------------
    # Print top 10 candidates for each method
    # ------------------------------------------------------------------
    methods = {
        "Logistic Regression": lr_scores,
        "Random Forest": rf_scores,
        "RGCN": rgcn_scores,
        "Path Counting": path_counts,
        "Weighted PageRank": ppr_scores
    }

    print("\n" + "=" * 80)
    print(f"Top 10 drug candidates for {matched_disease} by each method")
    print("=" * 80)
    for name, scores_dict in methods.items():
        if not scores_dict:
            print(f"\n{name}: No results")
            continue
        sorted_items = sorted(scores_dict.items(), key=lambda x: x[1], reverse=True)[:10]
        print(f"\n{name}:")
        for i, (drug, score) in enumerate(sorted_items, 1):
            if isinstance(score, float):
                print(f"  {i:2d}. {drug}: {score:.6f}")
            else:
                print(f"  {i:2d}. {drug}: count={score}")

    # ------------------------------------------------------------------
    # PubMed validation (optional, requires email)
    # ------------------------------------------------------------------
    # Uncomment and set your email to use PubMed validation
    # Entrez.email = "your_email@example.com"
    #
    # candidate_drugs = set()
    # for scores_dict in [lr_scores, rf_scores, rgcn_scores, path_counts, ppr_scores]:
    #     if scores_dict:
    #         top10 = sorted(scores_dict.items(), key=lambda x: x[1], reverse=True)[:10]
    #         for drug, _ in top10:
    #             candidate_drugs.add(drug)
    #
    # print(f"\nValidating {len(candidate_drugs)} unique candidate drugs with PubMed...")
    # for drug in candidate_drugs:
    #     query = f"({drug}) AND ({matched_disease})"
    #     # ... (implement PubMed search using Biopython)
    #     # See notebook for full implementation

if __name__ == "__main__":
    main()